## odin-sequencer integration demo notebook

#### This Jupyter notebook demonstrates use of the odin-sequencer remote procedure call (RPC) client to interact with a running sequencer

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

from odin_sequencer.rpc.client import OdinSequencerClient, OdinSequencerClientError

# Create a client connected to a sequencer running at the specified address. 
seq = OdinSequencerClient("127.0.0.1", emit_exceptions=False)

# Get a sequencer context object to interact with directly
test_device = seq.get_context("test_device")

### Execute a sequence by name with positional arguments

As well as returning the result of the sequence, the output of any `print()` statements in the sequence are captured and displayed during execution

In [ ]:
result = seq.execute("add", 1, 2)
print(f">>> Result is {result} of type {type(result)}")

### Sequences can also be called as methods of the client

In this case the arguments are being passed as explicit keywords

In [ ]:
result = seq.add(a=3, b=4)
print(f">>> Result is {result} of type {type(result)}")

### Similarly, context methods can be executed by name ...

In [ ]:
result = test_device.execute("read_reg")
print(f">>> Register value is {result}")

### ... or as methods of the context itself

In [ ]:
result = test_device.read_reg()
print(f">>> Register value is {result}")

### You can control how exceptions generated during sequence execution are handled.

Depending on the value of the emit_exceptions argument passed to OdinSequencerClient at initialisation, or through a call to `emit_exceptions()`,
exceptions generated by the server will either generate a client exception or simply print the error message

In [ ]:
seq.emit_exceptions(True)  # Try changing the value here
try:
    result = test_device.read_missing()
except OdinSequencerClientError as error:
    print(f"Exception raised: {error}")
else:
    print("No exception raised (error should be printed above)")

### Likewise for direct method calls to sequences

In [ ]:
seq.emit_exceptions(False)
seq.exceptional_add(a=7, b=35)

### Long-running sequences can be aborted

This requires the standard mechansim, i.e. that the sequence calls `abort_sequence()` check regularly. In a Jupyter notebook, the sequence can be aborted by
interrupting the kernel from the menu above, or with the keyboard shortcut `I, I` (i.e. pressing the "I" key twice)

In [ ]:
seq.execute("abortable_sequence")

### The sequencer can be instructed to reload currently loaded sequences

In [ ]:
seq.reload()

### The returned result of executing a sequence can also be a numpy array for e.g. image display

In [ ]:
result = seq.execute("array_test", shape=[1024,1024])
_ = plt.imshow(result)

### Multiple results, including numpy arrays, can also be returned

In [ ]:
t, signal = seq.sine_test(freq=5, duration=1, rate=1000)
plt.plot(t, signal)
plt.xlabel("Time [s]")
plt.ylabel("Amplitude")
plt.grid(True)